# Throwaway Forecasting Branch

Demonstrates both branch use cases:
- **dev** branch: long-lived development iteration (DDL, schema changes, promotion)
- **forecast-cold-snap** branch: throwaway forecasting scenario (create → use → delete)

Executed: 2026-08-28

In [1]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Branch, BranchSpec, Duration

w = WorkspaceClient()
PROJECT = 'northpeak'
BRANCH_ID = 'forecast-cold-snap'

print(f'Creating throwaway forecasting branch: {BRANCH_ID}')
print(f'  Source: projects/{PROJECT}/branches/production')
print(f'  TTL: 4 hours (auto-deletes after forecast scenario)')

op = w.postgres.create_branch(
    parent=f'projects/{PROJECT}',
    branch=Branch(spec=BranchSpec(
        source_branch=f'projects/{PROJECT}/branches/production',
        ttl=Duration(seconds=14400),  # 4 hours — throwaway
    )),
    branch_id=BRANCH_ID,
)
branch = op.wait()
print(f'\n✓ Branch created: {branch.name}')
print(f'  State: {branch.status.current_state}')

Creating throwaway forecasting branch: forecast-cold-snap
  Source: projects/northpeak/branches/production
  TTL: 4 hours (auto-deletes after forecast scenario)

✓ Branch created: projects/northpeak/branches/forecast-cold-snap
  State: BranchStatusState.READY


In [2]:
import itertools

# Get endpoint for the throwaway branch
endpoints = list(itertools.islice(
    w.postgres.list_endpoints(parent=f'projects/{PROJECT}/branches/{BRANCH_ID}'), 10
))
print(f'Endpoints on throwaway branch \'{BRANCH_ID}\':')
for ep in endpoints:
    print(f'  {ep.name}')
    print(f'  Host: {ep.status.hosts.host}')
    print(f'  State: {ep.status.current_state}')
    forecast_host = ep.status.hosts.host

Endpoints on throwaway branch 'forecast-cold-snap':
  projects/northpeak/branches/forecast-cold-snap/endpoints/primary
  Host: ep-jolly-breeze-d2c44ucy.database.us-east-1.cloud.databricks.com
  State: EndpointStatusState.ACTIVE


In [3]:
import psycopg2, psycopg2.extras

# Connect to throwaway forecasting branch
cred = w.postgres.generate_database_credential(
    endpoint=f'projects/{PROJECT}/branches/{BRANCH_ID}/endpoints/primary'
)
username = w.current_user.me().user_name

conn_forecast = psycopg2.connect(
    host=forecast_host, database='databricks_postgres',
    user=username, password=cred.token, port=5432, sslmode='require'
)
conn_forecast.autocommit = True
print(f'Connected to throwaway branch: {BRANCH_ID}')
print(f'  Host: {forecast_host}')

# Run cold-snap forecasting scenario: 2x demand for cold_weather in North zone
with conn_forecast.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        WITH latest_snap AS (
            SELECT MAX(snapshot_date) AS max_date FROM northpeak.synced_inventory
        ),
        forecast AS (
            SELECT i.store_id, i.store_name, i.city, i.region,
                   i.product_id, i.product_name, i.on_hand_units, i.price_usd,
                   COALESCE((SELECT AVG(i2.on_hand_units) FROM northpeak.synced_inventory i2
                     WHERE i2.store_id = i.store_id AND i2.product_id = i.product_id
                     AND i2.snapshot_date >= (SELECT max_date - INTERVAL '7 days' FROM latest_snap)
                   ), 5) * 2 AS forecast_daily_demand,
                   CASE WHEN i.on_hand_units > 0
                        THEN ROUND(i.on_hand_units / (COALESCE(
                           (SELECT AVG(i2.on_hand_units) FROM northpeak.synced_inventory i2
                            WHERE i2.store_id = i.store_id AND i2.product_id = i.product_id
                            AND i2.snapshot_date >= (SELECT max_date - INTERVAL '7 days' FROM latest_snap)
                           ), 5) * 2)::numeric, 1)
                        ELSE 0 END AS days_until_stockout
            FROM northpeak.synced_inventory i
            CROSS JOIN latest_snap ls
            WHERE i.snapshot_date = ls.max_date
              AND i.climate_zone = 'North' AND i.seasonality = 'cold_weather'
        )
        SELECT store_id, store_name, city, product_name,
               on_hand_units, forecast_daily_demand, days_until_stockout,
               ROUND((forecast_daily_demand * 14 - on_hand_units) * price_usd) AS forecast_exposure_usd
        FROM forecast WHERE days_until_stockout < 5
        ORDER BY forecast_exposure_usd DESC LIMIT 10
    """)
    results = cur.fetchall()

print(f'\nCOLD-SNAP FORECASTING SCENARIO (throwaway branch: {BRANCH_ID})')
print(f'Scenario: 2x demand multiplier for cold_weather products in North zone')
print(f'Results: {len(results)} stores projected to stockout within 5 days')
print('-' * 80)
for r in results:
    print(f"  {r['store_id']:<12} {r['city']:<18} {r['product_name']:<25} on_hand={r['on_hand_units']:>3}  days_left={r['days_until_stockout']}  exposure=${r['forecast_exposure_usd']:,.0f}")
print('\nThis forecast ran on the throwaway branch — production data unchanged.')

Connected to throwaway branch: forecast-cold-snap
  Host: ep-jolly-breeze-d2c44ucy.database.us-east-1.cloud.databricks.com

COLD-SNAP FORECASTING SCENARIO (throwaway branch: forecast-cold-snap)
Scenario: 2x demand multiplier for cold_weather products in North zone
Results: 10 stores projected to stockout within 5 days
--------------------------------------------------------------------------------
  STORE-0014   Pittsburgh         Boot 481                  on_hand=139  days_left=0.5  exposure=$974,654
  STORE-0101   Denver             Boot 481                  on_hand=139  days_left=0.5  exposure=$974,654
  STORE-0284   Detroit            Chino 203                 on_hand=139  days_left=0.5  exposure=$974,542
  STORE-0138   Portland           Chino 203                 on_hand=139  days_left=0.5  exposure=$974,542
  STORE-0384   Chicago            Boot 481                  on_hand=138  days_left=0.5  exposure=$967,642
  STORE-0280   Milwaukee          Boot 481                  on_hand=1

In [4]:
# Delete the throwaway branch — full lifecycle complete
conn_forecast.close()

# Show all branches before deletion
branches = list(w.postgres.list_branches(parent=f'projects/{PROJECT}'))
print('All branches:')
for b in branches:
    tag = '← THROWAWAY (4hr TTL)' if 'forecast' in b.name else ''
    print(f"  {b.name.split('/')[-1]:<25} state={b.status.current_state}  {tag}")

# Delete
print(f'\nDeleting throwaway branch: {BRANCH_ID}')
w.postgres.delete_branch(name=f'projects/{PROJECT}/branches/{BRANCH_ID}')
print('  ✓ Throwaway branch deleted successfully')

print('\nBranch lifecycle complete:')
print('  1. Created forecast-cold-snap (TTL=4h, copy-on-write from production)')
print('  2. Ran cold-snap demand forecast (2x multiplier) — 10 at-risk stores')
print('  3. Deleted branch — production unchanged, zero residual cost')

All branches:
  forecast-cold-snap        state=BranchStatusState.READY  ← THROWAWAY (4hr TTL)
  dev                       state=BranchStatusState.READY  
  production                state=BranchStatusState.READY  
  dev-inventory-forecast    state=BranchStatusState.READY  ← THROWAWAY (4hr TTL)

Deleting throwaway branch: forecast-cold-snap
  ✓ Throwaway branch deleted successfully

Branch lifecycle complete:
  1. Created forecast-cold-snap (TTL=4h, copy-on-write from production)
  2. Ran cold-snap demand forecast (2x multiplier) — 10 at-risk stores
  3. Deleted branch — production unchanged, zero residual cost
